# SAM3 Polygon Annotation Template (Colab)

This notebook demonstrates automatic polygon annotation for any type of images using Meta's SAM3 model.

**Features:**
- Pure SAM3 + NumPy implementation (no OpenCV)
- Text-based segmentation prompts
- Automatic polygon generation from masks
- JSON export for annotations
- Visual results with bounding boxes and polygons

In [ ]:
# ============================================================
#  PARAMETERS — Change these values before running the notebook
# ============================================================

# --- Paths (Google Drive) ---
image_folder_path = "/content/drive/MyDrive/Input_Images"       # Input folder on Drive
output_folder     = "/content/drive/MyDrive/Output_Annotations"  # Output folder on Drive

# --- Model ---
MODEL_NAME = "facebook/sam3"   # SAM3 model name on HuggingFace

# --- Segmentation ---
text_prompt         = "magic_wand"  # Object to detect (e.g. "cat", "pizza", "headline", "newspaper")
DETECTION_THRESHOLD = 0.3           # Detection confidence (0.1 – 0.5)
MASK_THRESHOLD      = 0.3           # Mask quality (0.1 – 0.5)
MIN_MASK_AREA       = 50            # Skip masks smaller than this (pixels)

# --- Image Enhancement ---
CONTRAST_FACTOR  = 1.2   # Contrast boost (1.0 = no change)
SHARPNESS_FACTOR = 1.1   # Sharpness boost (1.0 = no change)

# --- Supported image extensions ---
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".gif", ".bmp", ".tiff")

## Step 1: Install Dependencies

In [ ]:
# Install the latest transformers with SAM3 support
!pip install git+https://github.com/huggingface/transformers
!pip install torch torchvision pillow numpy

: 

## Step 2: Authentication Setup

In [ ]:
# Login to HuggingFace to access the gated SAM3 model
from huggingface_hub import login
login()  # You'll need to enter your HF token

## Step 3: Import Required Libraries

In [ ]:
import numpy as np
import json
import os
from PIL import Image, ImageEnhance
from transformers import Sam3Processor, Sam3Model
import torch
from datetime import datetime

# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print("Ready for SAM3-based polygon annotation!")

## Step 4: Initialize SAM3 Model

In [ ]:
# Load SAM3 model and processor
print("Loading SAM3 model...")
model = Sam3Model.from_pretrained(MODEL_NAME).to(device)
processor = Sam3Processor.from_pretrained(MODEL_NAME)

print("SAM3 model loaded successfully!")

## Step 5: Define Polygon Extraction Function

In [ ]:
def mask_to_rle(mask):
    """Convert binary mask to RLE (Run-Length Encoding) format"""
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return runs.tolist()

def enhance_image_quality(image):
    """Enhance image for better target object detection"""
    enhancer = ImageEnhance.Contrast(image)
    image = enhancer.enhance(CONTRAST_FACTOR)
    enhancer = ImageEnhance.Sharpness(image)
    image = enhancer.enhance(SHARPNESS_FACTOR)
    return image

def create_mask_annotations(image, results, text_prompt):
    """Create mask-based annotations with visualization"""

    mask_annotations = []

    colors = [
        (255, 0, 0, 180), (0, 255, 0, 180), (0, 0, 255, 180),
        (255, 255, 0, 180), (255, 0, 255, 180), (0, 255, 255, 180),
        (255, 128, 0, 180), (128, 0, 255, 180), (255, 192, 203, 180),
        (128, 128, 128, 180)
    ]

    mask_overlay = np.zeros((image.height, image.width, 4), dtype=np.uint8)

    print(f"Creating mask annotations for {len(results['masks'])} objects...")

    for i, (mask, box, score) in enumerate(zip(results['masks'], results['boxes'], results['scores'])):
        print(f"   Processing mask {i+1}/{len(results['masks'])} (score: {score:.3f})")

        if torch.is_tensor(mask):
            mask_np = mask.cpu().numpy()
        else:
            mask_np = mask

        binary_mask = (mask_np > 0.5).astype(np.uint8)

        mask_area = int(np.sum(binary_mask))
        if mask_area < MIN_MASK_AREA:
            continue

        if torch.is_tensor(box):
            bbox = box.tolist()
        else:
            bbox = box

        rle = mask_to_rle(binary_mask)

        mask_annotation = {
            "id": i,
            "category": text_prompt,
            "mask_rle": rle,
            "bbox": bbox,
            "score": float(score),
            "area": mask_area,
            "mask_shape": [int(image.height), int(image.width)]
        }
        mask_annotations.append(mask_annotation)

        color = colors[i % len(colors)]
        mask_colored = np.zeros((image.height, image.width, 4), dtype=np.uint8)
        mask_colored[binary_mask == 1] = color
        mask_overlay = np.maximum(mask_overlay, mask_colored)

        x1, y1, x2, y2 = [int(coord) for coord in bbox]
        print(f"      Area: {mask_area:,} pixels")
        print(f"      Bbox: [{x1}, {y1}, {x2}, {y2}]")
        print(f"      RLE size: {len(rle)} values")

    mask_overlay_img = Image.fromarray(mask_overlay, 'RGBA')
    mask_result = Image.alpha_composite(image.convert('RGBA'), mask_overlay_img)

    return {
        'annotations': mask_annotations,
        'overlay_image': mask_result.convert('RGB'),
        'mask_overlay': mask_overlay_img
    }

print("Mask annotation functions defined")

## Step 6: Load Images

In [ ]:
# Mount Google Drive and set up paths
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(output_folder, exist_ok=True)

if not os.path.exists(image_folder_path):
    print(f"Error: Input folder does not exist: {image_folder_path}")
    print("Please update image_folder_path in the PARAMETERS cell.")
else:
    print(f"Input folder found: {image_folder_path}")

    image_files = [
        f for f in os.listdir(image_folder_path)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    ]

    if not image_files:
        print(f"No image files found in {image_folder_path}")
    else:
        print(f"Found {len(image_files)} images in {image_folder_path}")
        print("Images found:", image_files)
        print("Ready for batch processing!")

## Step 7: Configure and Run SAM3 Segmentation

In [ ]:
if 'image_files' in locals() and len(image_files) > 0:
    print(f"Starting batch processing of {len(image_files)} images...")
    print(f"Creating masks for: '{text_prompt}'")

    processed_count = 0
    successful_count = 0
    failed_images = []

    for i, img_name in enumerate(image_files, 1):
        current_image_path = os.path.join(image_folder_path, img_name)
        print(f"\n--- Processing {i}/{len(image_files)}: {img_name} ---")

        try:
            original_image = Image.open(current_image_path).convert("RGB")
            image = enhance_image_quality(original_image)

            print(f"   Image size: {image.size}")
            print("   Running SAM3 inference...")

            inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model(**inputs)

            results = processor.post_process_instance_segmentation(
                outputs,
                threshold=DETECTION_THRESHOLD,
                mask_threshold=MASK_THRESHOLD,
                target_sizes=inputs.get("original_sizes").tolist()
            )[0]

            if len(results['masks']) > 0:
                mask_results = create_mask_annotations(image, results, text_prompt)

                print(f"   Created {len(mask_results['annotations'])} mask annotations")

                base_name = os.path.splitext(img_name)[0]
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

                mask_data = {
                    "metadata": {
                        "image_path": current_image_path,
                        "image_size": [image.width, image.height],
                        "model": MODEL_NAME,
                        "annotation_type": "masks_only",
                        "prompt": text_prompt,
                        "timestamp": timestamp,
                        "encoding": "RLE (Run-Length Encoding)",
                        "thresholds": {
                            "detection_threshold": DETECTION_THRESHOLD,
                            "mask_threshold": MASK_THRESHOLD
                        }
                    },
                    "masks": mask_results['annotations'],
                    "summary": {
                        "total_masks": len(mask_results['annotations']),
                        "avg_score": np.mean([ann['score'] for ann in mask_results['annotations']]) if mask_results['annotations'] else 0,
                        "total_area": sum([ann['area'] for ann in mask_results['annotations']]),
                        "coverage_percent": (sum([ann['area'] for ann in mask_results['annotations']]) / (image.width * image.height)) * 100,
                        "avg_mask_size": np.mean([ann['area'] for ann in mask_results['annotations']]) if mask_results['annotations'] else 0
                    }
                }

                json_path = os.path.join(output_folder, f"{base_name}_{text_prompt}_masks.json")
                with open(json_path, 'w') as f:
                    json.dump(mask_data, f, indent=2)
                print(f"   Saved JSON: {os.path.basename(json_path)}")

                overlay_img_path = os.path.join(output_folder, f"{base_name}_{text_prompt}_mask_overlay.jpg")
                mask_results['overlay_image'].save(overlay_img_path, quality=95)
                print(f"   Saved overlay: {os.path.basename(overlay_img_path)}")

                pure_mask_path = os.path.join(output_folder, f"{base_name}_{text_prompt}_masks_only.png")
                mask_results['mask_overlay'].save(pure_mask_path)
                print(f"   Saved masks: {os.path.basename(pure_mask_path)}")

                successful_count += 1
            else:
                print(f"   No masks found for {img_name}. Skipping.")
                failed_images.append((img_name, "No masks detected"))

            processed_count += 1

        except FileNotFoundError:
            error_msg = f"Image file not found: {current_image_path}"
            print(f"   {error_msg}")
            failed_images.append((img_name, error_msg))
        except Exception as e:
            error_msg = f"Processing error: {str(e)}"
            print(f"   {error_msg}")
            failed_images.append((img_name, error_msg))

    print(f"\n" + "="*60)
    print(f"BATCH PROCESSING COMPLETE")
    print(f"="*60)
    print(f"Total images: {len(image_files)}")
    print(f"Successful: {successful_count}")
    print(f"Failed: {len(failed_images)}")
    print(f"Output folder: {output_folder}")

    if failed_images:
        print(f"\nFailed images:")
        for img_name, error in failed_images:
            print(f"   - {img_name}: {error}")

    print(f"\nPrompt used: '{text_prompt}'")

else:
    print("No images found to process. Please run the previous cell first.")

## Step 8: Save Results

In [ ]:
print("USAGE INSTRUCTIONS")
print("=" * 50)
print()
print("SETUP:")
print("1. Change parameters in the PARAMETERS cell at the top")
print("2. Run all cells from Step 1-6 to set up the environment")
print()
print("BATCH PROCESSING:")
print("3. Run Step 7 to process ALL images in your folder")
print()
print("PROMPT OPTIONS:")
print("- 'target_object' - Detect entire target_object regions")
print("- 'headline' - Detect article headlines")
print("- 'text' - Detect general text regions")
print("- 'paragraph' - Detect text paragraphs")
print("- 'article' - Detect complete articles")
print("- 'image' - Detect photos/graphics")
print("- 'caption' - Detect image captions")
print("- 'advertisement' - Detect ad sections")
print()
print("OUTPUT FILES (for each image):")
print("- {image_name}_{prompt}_masks.json - Annotation data with RLE masks")
print("- {image_name}_{prompt}_mask_overlay.jpg - Original image with colored masks")
print("- {image_name}_{prompt}_masks_only.png - Pure mask visualization")
print()
print("CUSTOMIZATION:")
print("- Change 'text_prompt' in PARAMETERS cell to use different prompts")
print("- Adjust 'DETECTION_THRESHOLD' (0.1-0.5) for detection sensitivity")
print("- Modify 'MASK_THRESHOLD' (0.1-0.5) for mask quality")

## Step 9: Experiment with Different Prompts (Optional)

## Usage Notes

**Prompts to try:**
- `"text"` - General text regions
- `"headline"` - Article headlines
- `"paragraph"` - Text paragraphs
- `"article"` - Complete articles
- `"image"` - Photos/graphics
- `"caption"` - Image captions
- `"advertisement"` - Ad sections

**Output Files:**
- `{image_name}_{prompt}_annotations.json` - Polygon data
- `{image_name}_{prompt}_annotated.jpg` - Visual results

**Customization:**
- Adjust `threshold` and `mask_threshold` in step 7 for sensitivity
- Modify `sample_density` in step 5 for polygon detail
- Change visualization colors in step 8